### Loading the dataset

In [2]:
with open("../charwise-gpt/dickens/combined.txt", "r", encoding='utf-8') as f:
    text = f.read()

print(text[:1000])      
print(f"length of dataset in chars: {len(text)}")

The Project Gutenberg eBook, Three Ghost Stories, by Charles Dickens


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.org





Title: Three Ghost Stories


Author: Charles Dickens



Release Date: March 9, 2013  [eBook #1289]
[This file was first posted on April 5, 1998]

Language: English

Character set encoding: UTF-8


***START OF THE PROJECT GUTENBERG EBOOK THREE GHOST STORIES***


Transcribed from the 1894 Chapman and Hall edition of “Christmas Stories”
by David Price, email ccx074@pglaf.org





                           THREE GHOST STORIES


                            by Charles Dickens




CONTENTS

The Haunted House             121
The Trial For Murder          303
The Signal-Man                312




THE HAUNTED HOUSE.
IN TWO CHAPTERS. {121}


                                 [


### Building character tokens    

In [3]:
characters = sorted(list(set(text)))
print(f"characters: {''.join(characters)}")
vocab_size = len(characters)
print(f"vocab size: {vocab_size}")

# first time using those functions
# enumerate returns a list?/object containing pairs of number/value
stoi = {ch:i for i, ch in enumerate(characters)}
# we're creating two dicts dynamically, in a:b a is the key and b is the value
itos = {i:ch for i, ch in enumerate(characters)}

encode = lambda s: [stoi[c] for c in s] # encode a string, looping through its char elts
decode = lambda s: ''.join(itos[c] for c in s) # take a list of ints, output a string

print(f"encoding of hello world: f{encode('hello world')}")
print(f"decoding of [69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]: {decode([69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65])}")
print(f"sanity check: {decode(encode(''.join(characters)))}")

characters: 
 !"#$%&'()*,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz{}~ £½Ôàáâæçèéêëíîòóôöāěŏœ—‘’“”﻿
vocab size: 120
encoding of hello world: f[69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]
decoding of [69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]: hello world
sanity check: 
 !"#$%&'()*,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz{}~ £½Ôàáâæçèéêëíîòóôöāěŏœ—‘’“”﻿


### Storing into a tensor

In [4]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([24350792]) torch.int64
tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1,  39,
         82,  81,  66,  75,  63,  66,  79,  68,   1,  66,  34,  76,  76,  72,
         12,   1,  52,  69,  79,  66,  66,   1,  39,  69,  76,  80,  81,   1,
         51,  81,  76,  79,  70,  66,  80,  12,   1,  63,  86,   1,  35,  69,
         62,  79,  73,  66,  80,   1,  36,  70,  64,  72,  66,  75,  80,   0,
          0,   0,  52,  69,  70,  80,   1,  66,  34,  76,  76,  72,   1,  70,
         80,   1,  67,  76,  79,   1,  81,  69,  66,   1,  82,  80,  66,   1,
         76,  67,   1,  62,  75,  86,  76,  75,  66,   1,  62,  75,  86,  84,
         69,  66,  79,  66,   1,  62,  81,   1,  75,  76,   1,  64,  76,  80,
         81,   1,  62,  75,  65,   1,  84,  70,  81,  69,   0,  62,  73,  74,
         76,  80,  81,   1,  75,  76,   1,  79,  66,  80,  81,  79,  70,  64,
         81,  70,  76,  75,  80,   1,  84,  69,  62,  81,  80,  76,  66,  83,
         66,  79,  14,   1,  

### Train/val separation

In [5]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

### Block setup

In [6]:
block_size = 16
train_data[:block_size+1]

# all possible examples:
# x as input, y as target
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    print(f"when context: {x[:t+1]}, target: {y[t]}")


when context: tensor([119]), target: 52
when context: tensor([119,  52]), target: 69
when context: tensor([119,  52,  69]), target: 66
when context: tensor([119,  52,  69,  66]), target: 1
when context: tensor([119,  52,  69,  66,   1]), target: 48
when context: tensor([119,  52,  69,  66,   1,  48]), target: 79
when context: tensor([119,  52,  69,  66,   1,  48,  79]), target: 76
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76]), target: 71
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71]), target: 66
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66]), target: 64
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64]), target: 81
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81]), target: 1
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1]), target: 39
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1,  39])

### Dataloader

In [7]:
block_size = 16 #context length
batch_size = 4 #independent sequences to process in parallel

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # here, params are high (upper limit for sampling) and size, that is the number of elts to sample
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    # target range, which is why we're not just sampling 1 at a time
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
    
xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

for b in range (batch_size):
    for t in range (block_size):
        print(f"when context is {xb[b, :t+1]} target is {yb[b, t]}")

inputs:
torch.Size([4, 16])
tensor([[  1,  62,  80,  72,  66,  65,  14,   0,   0,   3,  55,  69,  86,  12,
           1,  73],
        [ 75,  76,  70,  80,  66,   1,  62,  75,  65,   1,  79,  70,  76,  81,
           0,  70],
        [ 66,  75,  81,  73,  66,  74,  62,  75,  12,   1,  80,  70,  79,  31,
         116,   0],
        [ 76,  82,   1,  74,  70,  68,  69,  81,   1,  75,  76,  81,  12,   1,
          69,  76]])
targets:
torch.Size([4, 16])
tensor([[ 62,  80,  72,  66,  65,  14,   0,   0,   3,  55,  69,  86,  12,   1,
          73,  70],
        [ 76,  70,  80,  66,   1,  62,  75,  65,   1,  79,  70,  76,  81,   0,
          70,  75],
        [ 75,  81,  73,  66,  74,  62,  75,  12,   1,  80,  70,  79,  31, 116,
           0,   0],
        [ 82,   1,  74,  70,  68,  69,  81,   1,  75,  76,  81,  12,   1,  69,
          76,  84]])
when context is tensor([1]) target is 62
when context is tensor([ 1, 62]) target is 80
when context is tensor([ 1, 62, 80]) target is 72
when context

### Simplest possible nn : bigram

In [8]:
# cross entropy / negative log likelihood
import math

def CEL(pred: list[float], true_idx: int):
    sum_exps = sum(math.exp(c) for c in pred)
    # compute softmax prob of the true class
    prob_true = math.exp(pred[true_idx])/sum_exps
    return -math.log(prob_true)

# tests
print(CEL([0.0, -100.0, -100.0], 0))
print(CEL([0.1, 2, 0.3], 1))
print(CEL([0.1, 0.2, 0.3], 2))

-0.0
0.28687085095710846
1.001942848229244


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lut
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (batch_size (B), block_size (T)) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T, vocab_size (C)), this is the same as [idx], accessing the idxth row
        # pytorch expects a two dimensional object for the loss, i.e. instance * vocab_size
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # __call__ is defined as forward in nn.Module ?
            logits = logits[:, -1, :] # takes the last elt in Time (T) / context dim
            probs = F.softmax(logits, dim=-1) # converts to probs
            idx_next = torch.multinomial(probs, num_samples=1) # sample from the distribution
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx              

m = BigramLanguageModel(vocab_size=vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([64, 120])
tensor(5.1461, grad_fn=<NllLossBackward0>)

UI,W>ŏEè'CICzvKpUBGjò 'v[.$ç- et'gáâ}Tî﻿gí﻿po'nn:=Ve@?_pY&J7ío]~HoN’óDM#n_IöBrTpDá5£=S)f0pêq7BrN6UKö


### Training the bigram, optimizer

In [14]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())

2.523918628692627
2.5104258060455322
2.545370578765869
2.469841718673706
2.56523060798645
2.532658100128174
2.4399328231811523
2.430236577987671
2.493853807449341
2.4755492210388184
2.531493902206421
2.443241596221924
2.5555412769317627
2.4223849773406982
2.3807480335235596
2.5696187019348145
2.5622401237487793
2.5109870433807373
2.441114664077759
2.567394256591797
2.5023932456970215
2.5355498790740967
2.426485300064087
2.530008316040039
2.479815721511841
2.425309181213379
2.4828121662139893
2.5331974029541016
2.459536552429199
2.5350353717803955
2.5009565353393555
2.5326836109161377
2.476130247116089
2.476964235305786
2.5689198970794678
2.518744468688965
2.4815175533294678
2.5191967487335205
2.5795602798461914
2.4947474002838135
2.491431474685669
2.435368776321411
2.507319688796997
2.4649276733398438
2.5190417766571045
2.5149800777435303
2.4686310291290283
2.4486451148986816
2.5198845863342285
2.539430618286133
2.5303468704223633
2.5249853134155273
2.4994399547576904
2.448816299438476